In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings 

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

c:\Users\User\Documents\SabioGroup\rag-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)


True

In [4]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings) #Chroma class always returns objects with a .get() method, which allows us to retrieve specific vectors from the collection. By using the .get() method, we can specify parameters such as limit and include to control how many vectors we want to retrieve and what information we want to include in the returned data. This is useful for examining the structure of the vectors in our vector store and understanding how they are organized for later retrieval when we query the knowledge base. Those objects have other functionalities as well, such as adding new vectors, performing similarity searches, and managing the collection of vectors in the vector store. By using the Chroma class, we can effectively manage our vector store and utilize it for efficient retrieval of relevant information when we query our knowledge base later on.
# e.g collection = vectorstore.get_collection() # The get_collection() method of the Chroma vector store

In [5]:
retriever = vectorstore.as_retriever() #it will run as a retriever, which allows us to perform similarity searches and retrieve relevant document chunks based on a query. By using the as_retriever() method, we can leverage the capabilities of the Chroma vector store to find and return the most relevant chunks of information from our knowledge base when we later query it with user input. This is an essential step in building a retrieval-augmented generation (RAG) system, where we want to combine the power of a language model with the ability to retrieve specific information from a large collection of documents. It know where to go and what is stored on the db_name (embedding database) and how to retrieve the relevant information when we query it later on.
llm = ChatOpenAI(temperature=0, model_name=MODEL) #the LLM (Language Model) is initialized with the specified temperature and model name. The temperature parameter controls the randomness of the model's output, with lower values making the output more deterministic and higher values making it more creative. By setting the temperature to 0, we are instructing the model to generate more focused and consistent responses, which can be beneficial for tasks that require accuracy and reliability. The model_name parameter specifies which version of the language model we want to use, in this case, "gpt-4.1-nano". This allows us to leverage the capabilities of that specific model for generating responses based on the retrieved information from our vector store.

In [6]:
retriever.invoke("Who is Avery?")  #the retriever's invoke method is called with the query "Who is Avery?". This method will perform a similarity search in the vector store to find and retrieve relevant document chunks that are related to the query. The retrieved information will then be used as context for the language model (LLM) to generate a response based on the information found in the vector store. By invoking the retriever with this query, we can test how well it retrieves relevant information about "Avery" from our knowledge base, which can help us understand the effectiveness of our retrieval-augmented generation (RAG) system.

[Document(id='a1608c5d-22fe-4004-82b7-d407c3925718', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the

In [7]:
llm.invoke("Who is Avery?") #here the LLM will generate a response based on the retrieved information from the vector store about "Avery". The LLM will use the context provided by the retriever to formulate a coherent and relevant answer to the query "Who is Avery?", which may include details about Avery's background, characteristics, or any other relevant information that was retrieved from the vector store. This demonstrates how the LLM can utilize the retrieved information to generate informed responses to user queries. At the moment the llm is not properly connected to the retriever, so it will not be able to generate a response based on the retrieved information until we establish that connection in our RAG system. This is an important step in building a functional RAG system, as it allows the LLM to leverage the retrieved information to provide accurate and contextually relevant responses to user queries. Once we connect the LLM to the retriever, we can expect it to generate more informed and accurate responses based on the information retrieved from the vector store, which will enhance the overall performance of our RAG system.

AIMessage(content='Avery is a name that can refer to various individuals, characters, or entities depending on the context. Could you please provide more details or specify which Avery you are referring to?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 11, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_8310402cf4', 'id': 'chatcmpl-Do84FFrH1EcyhH9ayr965zOkJZLjk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ea252-9096-7090-91d5-01099e98795d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 36, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0}

In [ ]:
llm.invoke("What is the capital of France?") #this will result in a proper correct answer, which is "Paris". The language model (LLM) will use its pre-trained knowledge to generate a response based on the query "What is the capital of France?" and provide the correct answer without needing to retrieve any additional information from the vector store, as this is a well-known fact that the model has been trained on.

In [8]:
# Time to put this together! With the retriever and LLM set up, we can now create a system prompt template that will guide the LLM in generating responses based on the retrieved information from the vector store. The system prompt will provide instructions to the LLM on how to use the retrieved context to answer user queries accurately and effectively. By crafting a well-designed system prompt, we can ensure that the LLM understands how to utilize the retrieved information to generate relevant and informative responses, which is crucial for the success of our RAG system. The system prompt will also include guidelines for the LLM to follow when generating responses, such as being honest about what it knows and does not know, and using the retrieved context appropriately to answer user queries.
SYSTEM_PROMPT_TEMPLATE = """
You are InsureLLM, a knowledgeable and friendly AI assistant representing the company InsureLLM.

Your role is to help users by providing accurate, clear, and helpful information about the company and its services.

Use the provided context when it is relevant to the user's question. If the answer cannot be found in the context or you are unsure, honestly state that you do not know rather than making up information.

Context:
{context}
"""

In [10]:
def answer_question(question: str, history):    #This function takes a user question and a history of previous interactions as input. It uses the retriever to fetch relevant documents from the vector store based on the user's question, and then constructs a system prompt using the retrieved context. The system prompt is designed to guide the language model (LLM) in generating an accurate and helpful response to the user's question. The LLM is then invoked with the system prompt and the user's question, and it generates a response based on the provided context. This function is a key component of our RAG system, as it allows us to leverage both retrieval and generation capabilities to provide informed answers to user queries.
    docs = retriever.invoke(question) #The retriever's invoke method is called with the user's question as the argument. This method will perform a similarity search in the vector store to find and retrieve relevant document chunks that are related to the user's question. The retrieved documents will then be used as context for the language model (LLM) to generate a response based on the information found in the vector store. By invoking the retriever with the user's question, we can ensure that the LLM has access to relevant information that can help it generate a more accurate and informed response to the user's query.
    context = "\n\n".join(doc.page_content for doc in docs) #The retrieved documents are processed to extract their content and combine it into a single context string. The page_content attribute of each retrieved document is accessed, and the contents are joined together with two newline characters ("\n\n") as a separator. This creates a cohesive context that can be provided to the language model (LLM) to help it generate a response based on the information found in the retrieved documents. By combining the retrieved document contents into a single context string, we can ensure that the LLM has access to all relevant information when generating its response to the user's question.
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context) #The system prompt is created by formatting the SYSTEM_PROMPT_TEMPLATE with the retrieved context. The format method is used to replace the {context} placeholder in the template with the actual context string that was generated from the retrieved documents. This system prompt will then be used to guide the language model (LLM) in generating a response that is informed by the retrieved information, ensuring that the LLM can provide accurate and relevant answers to the user's question based on the context provided.
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)]) #The language model (LLM) is invoked with a list of messages that includes a SystemMessage containing the system prompt and a HumanMessage containing the user's question. The LLM will process these messages to generate a response based on the instructions provided in the system prompt and the content of the user's question. The SystemMessage provides context and guidance to the LLM, while the HumanMessage represents the user's query that the LLM needs to answer. By invoking the LLM with these messages, we can expect it to generate a response that is informed by the retrieved context and addresses the user's question effectively.
    return response.content #The function returns the content of the response generated by the language model (LLM). The response is expected to be a string that contains the answer to the user's question, based on the context provided in the system prompt and the information retrieved from the vector store. By returning the response content, we can provide the user with a clear and informative answer to their query, leveraging both retrieval and generation capabilities in our RAG system.

In [12]:
answer_question("Who is Avery Lancaster?", [])

"Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, an insurance technology company she co-founded in 2015. She is based in San Francisco, California, and has played a key role in guiding the company's growth and innovation in the insurance industry. Avery has a background in product management and analytics, and she is known for her leadership, risk management expertise, and commitment to diversity and community engagement."

In [13]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
